## 数据聚合与分组操作  

1. 使用单键或多键(形式可以是函数、数组、DF列名)分割pandas对象

2. 计算分组汇总统计，计数、平均值、标准差或自定义函数

3. 组内转换，正态化、线性回归、排名或选取子集  

4. 计算透视表和交叉表  

5. 执行分位数分析及其他统计分析  

**groupby 参数(分组依据)**  
by:  

用于指定用于分组的列名、数组或其他分组键。可以是单个列名或列名的列表。  
示例：df.groupby("column_name") 或 df.groupby(["col1", "col2"])  



axis:  
    
指定分组的方向，0 表示按行分组（默认），1 表示按列分组。  
示例：df.groupby("column_name", axis=1)  



level:    
    
适用于多层索引的 DataFrame，指定要分组的层次级别。    
示例：df.groupby(level=0)   



as_index:    

布尔值，默认是 True，表示返回的结果是否使用分组的列作为行索引。如果设置为 False，分组列会作为普通列返回。   
示例：df.groupby("column_name", as_index=False).sum()   



group_keys:  
  
布尔值，默认是 True，表示在返回的结果中是否包括分组键。如果设置为 False，则返回的结果中不会包含分组键。   
示例：df.groupby("column_name", group_keys=False).sum() 



sort:  
   
布尔值，默认是 True，表示是否对分组键进行排序。设置为 False 可以提高性能，但结果的顺序可能会变化。   
示例：df.groupby("column_name", sort=False).sum()   



observed:   

适用于分类数据，布尔值，默认是 False。如果设置为 True，则只返回存在的组，而不是所有可能的组。  
示例：df.groupby("column_name", observed=True).sum()   

In [1]:
import numpy as np
import pandas as pd

**GroupBy**  
拆分——应用——联合



根据单键或多键拆分为多组，拆分操作一般在对象的特定轴上执行。  
如DF可以在其行(axis=index)或列(axis=columns)上进行分组。  


将函数用到各个分组，并产生新值，所有这些函数的执行操作会联合成为最终的结果对象。结果对象的形式一般取决于对数据所执行的操作。

In [3]:
df = pd.DataFrame({"key1": ["a", "a", None, "b", "b", "a", None],
                   "key2": pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
                   "data1": np.random.standard_normal(7),
                   "data2": np.random.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,-0.788180,-1.298039
1,a,2,-1.775482,0.910878
2,None,1,-0.269866,-0.641120
3,b,2,-1.466661,0.062179
4,b,1,-1.476165,1.140221
5,a,<NA>,0.172431,0.353426
6,None,1,-0.095394,-0.729769


In [13]:
# key1标签分组，并计算data1列的平均值。
grouped = df["data1"].groupby(df["key1"])   # 将 data1 列按 key1 列中的值进行分组
grouped

grouped只是一个特殊的GroupBy对象。它实际上还没有任何计算，只是包含了一些有关分组键df["key1"]的中间数据而已。  


人话：grouped这个对象里面已经含有对各分组执行运算所需的一切信息。  

In [15]:
# 可以调用GroupBy对象的mean方法计算分组平均值：
grouped.mean()

key1
a   -0.257583
b    0.585461
Name: data1, dtype: float64

数据先分割后聚合生成新的Series,索引为key1列中的唯一值。

In [17]:
# 一次传入包含多个数组的列表，会得到不同结果：
means = df["data1"].groupby([df["key1"], df["key2"]]).mean()
means

key1  key2
a     1       0.037861
      2      -1.311733
b     1       0.702981
      2       0.467942
Name: data1, dtype: float64

因为有两层索引，所以只会求唯一索引的平均值，有空缺值的求和也不排序.



![jupyter](10.1.png)

In [19]:
means.unstack()

key2,1,2
key1,,
a,0.037861,-1.311733
b,0.702981,0.467942


分组键均为Series，  

如果你使用单个 Series 作为分组键，数据会根据该 Series 中的唯一值进行分组。  
如果使用多个 Series，数据会根据这些 Series 的组合值进行分组。  



df["key1"] 和 df["key2"] 都是 Series，这意味着你在按这两个列的组合进行分组，从而生成一个多级索引的 DataFrame。

In [21]:
# 分组键可以是长度正确的任意数组：
states = np.array(["OH", "CA", "CA", "OH", "OH", "CA", "CA"])
years = [2005, 2005, 2006, 2005, 2006, 2005, 2006]

df["data1"].groupby([states, years]).mean()

CA  2005   -0.405305
    2006    0.898797
OH  2005    0.252901
    2006    0.702981
Name: data1, dtype: float64

相当于把这两列插入到原DF中，然后进行唯一值分组，如果数量对不上，就会填充为空值

In [7]:
states = np.array(["OH", "CA"])
years = [2005, 2005]

df["data1"].groupby([states, years]).mean(numeric_only=True)

ValueError: Grouper and axis must be same length

当groupby数组长度对不上时，会抛出valueerror

In [23]:
# 列名用作分组键,可以直接对该列内的内容自动分组计算
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,-0.257583,1.014844
b,1.5,0.585461,0.976514


In [13]:
df.groupby("key2").mean()

TypeError: agg function failed [how->mean,dtype->object]

df.groupby("key2").mean() 此行为在pandas2.0中已经更改   



可以添加.mean(numeric_only=True)保留旧行为

In [85]:
df.dtypes

key1      object
key2       Int64
data1    float64
data2    float64
dtype: object

TypeError: agg function failed [how->mean,dtype->object]    



类型错误，打印出 df.dtypes 第二列中除了int类型还有Na值，进行分组时不允许有两种数据类型，所以直接忽略第二列的Na值即可。

In [18]:
result = df.dropna(subset=['key2'])
print(result)
result.groupby("key2").mean(numeric_only=True)

   key1  key2     data1     data2
0     a     1 -0.788180 -1.298039
1     a     2 -1.775482  0.910878
2  None     1 -0.269866 -0.641120
3     b     2 -1.466661  0.062179
4     b     1 -1.476165  1.140221
6  None     1 -0.095394 -0.729769


,data1,data2
key2,,
1,-0.657401,-0.382177
2,-1.621071,0.486529


In [52]:
result = pd.DataFrame({"key1": ["a", "a", "a", "b", "b", "a", "b"],
                   "key2": pd.Series([1, 2, 1, 2, 1, 1, 1]),
                   "data1": np.random.standard_normal(7),
                   "data2": np.random.standard_normal(7)})
result

,key1,key2,data1,data2
0,a,1,-1.173914,-1.052885
1,a,2,-1.659514,0.974573
2,a,1,-0.206532,0.953968
3,b,2,-0.707295,-0.822035
4,b,1,-1.500237,1.297287
5,a,1,0.696575,-0.120615
6,b,1,0.178093,2.496691


In [20]:
result.groupby("key2").mean(numeric_only=True)

,data1,data2
key2,,
1,-0.657401,-0.382177
2,-1.621071,0.486529


In [54]:
result.dtypes

key1      object
key2       int64
data1    float64
data2    float64
dtype: object

In [29]:
df.groupby(["key1", "key2"]).mean()

data1     data2
key1 key2                    
a    1     0.037861  1.008009
     2    -1.311733  0.343925
b    1     0.702981  1.870438
     2     0.467942  0.082590

df.groupby("key2").mean() 若果能跑起来可能没有key1列，因为将key1列当作冗余列，直接移除了，默认情况下会聚合所有数值列。

In [99]:
# size方法可以返回包含分组大小的Series
df.groupby(["key1", "key2"]).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

In [101]:
# 缺失值默认被删除，可以忽略
df.groupby(["key1", "key2"], dropna=False).size()

key1  key2
a     1       1
      2       1
      <NA>    1
b     1       1
      2       1
NaN   1       2
dtype: int64

In [103]:
# 计算分组中非空值数量
df.groupby("key1").count()

,key2,data1,data2
key1,,,
a,2,3,3
b,2,2,2


#### 10.1.1 对分组进行迭代  
groupby 返回的对象支持迭代，可以产生一个二元组构成的序列，每个元组包含分组名和数据块

In [77]:
for name, group in df.groupby("key1"):
    print(name)
    print(group)

a
  key1  key2     data1     data2
0    a     1  0.037861  1.008009
1    a     2 -1.311733  0.343925
5    a  <NA>  0.501122  1.692599
b
  key1  key2     data1     data2
3    b     2  0.467942  0.082590
4    b     1  0.702981  1.870438


In [79]:
# 对于多个分组键的情况，元组的第一个元素是由键值组成的元组：
for (k1, k2), group in df.groupby(["key1", "key2"]):
    print((k1, k2))
    print(group)

('a', 1)
  key1  key2     data1     data2
0    a     1  0.037861  1.008009
('a', 2)
  key1  key2     data1     data2
1    a     2 -1.311733  0.343925
('b', 1)
  key1  key2     data1     data2
4    b     1  0.702981  1.870438
('b', 2)
  key1  key2     data1    data2
3    b     2  0.467942  0.08259


In [81]:
# 可以对分组的数据块进行任意操作
# 计算得到一个包含这些数据块的字典：就是将上面按key1分组的结果转化为字典
pieces = {name: group for name, group in df.groupby("key1")}
pieces

{'a':   key1  key2     data1     data2
 0    a     1  0.037861  1.008009
 1    a     2 -1.311733  0.343925
 5    a  <NA>  0.501122  1.692599,
 'b':   key1  key2     data1     data2
 3    b     2  0.467942  0.082590
 4    b     1  0.702981  1.870438}

In [83]:
pieces["b"]

,key1,key2,data1,data2
3,b,2,0.467942,0.082590
4,b,1,0.702981,1.870438


In [85]:
# 默认是在axis=index上进行的，也可以在其他轴
# 
grouped = df.groupby({"key1": "key", "key2": "key",
                      "data1": "data", "data2": "data"}, axis="columns")

for group_key, group_values in grouped:
    print(group_key)
    print(group_values)

data
      data1     data2
0  0.037861  1.008009
1 -1.311733  0.343925
2  1.226025 -0.321111
3  0.467942  0.082590
4  0.702981  1.870438
5  0.501122  1.692599
6  0.571569  0.169466
key
   key1  key2
0     a     1
1     a     2
2  None     1
3     b     2
4     b     1
5     a  <NA>
6  None     1


C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_4456\1312628352.py:3: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  grouped = df.groupby({"key1": "key", "key2": "key",


In [ ]:
grouped = df.groupby({"key1": "key", "key2": "key",
                      "data1": "data", "data2": "data"}, axis="columns")
这是一个字典，试图将列名映射到组名。
它将key1和key2列分到一个组（"key"），将data1和data2列分到另一个组（"data"）。
这种用法并不符合Pandas的groupby预期，通常不需要使用字典。

In [107]:
# 警告是弃用了axis=1的写法，要用转置写法
# 不过转置写法好像也不太对，但是总体效果差不多
df_T = df.T
grouped = df_T.groupby({"key1": "key", "key2": "key",
                        "data1": "data", "data2": "data"})

for group_key, group_values in grouped:
    print(group_key)
    print(group_values)

data
              0         1         2         3         4         5         6
data1  0.037861 -1.311733  1.226025  0.467942  0.702981  0.501122  0.571569
data2  1.008009  0.343925 -0.321111   0.08259  1.870438  1.692599  0.169466
key
      0  1     2  3  4     5     6
key1  a  a  None  b  b     a  None
key2  1  2     1  2  1  <NA>     1


In [109]:
grouped = df_T.groupby(df_T.index)
for group_key, group_values in grouped:
    print(group_key)
    print(group_values)

data1
              0         1         2         3         4         5         6
data1  0.037861 -1.311733  1.226025  0.467942  0.702981  0.501122  0.571569
data2
              0         1         2        3         4         5         6
data2  1.008009  0.343925 -0.321111  0.08259  1.870438  1.692599  0.169466
key1
      0  1     2  3  4  5     6
key1  a  a  None  b  b  a  None
key2
      0  1  2  3  4     5  6
key2  1  2  1  2  1  <NA>  1


#### 10.1.2 选取一列或多列  
使用单个列名或列名数组对由DF创建的Groupby对象建立索引，能实现选取部分列进行聚合效果：

In [113]:
# 根据key1、key2分组计算data2列的平均值并以DF返回结果：
df.groupby(["key1", "key2"])[["data2"]].mean()

data2
key1 key2          
a    1     1.008009
     2     0.343925
b    1     1.870438
     2     0.082590

In [135]:
# 传入数组或列表，则该索引操作返回的对象是一个经过分组的DF
# 如果传入的是标量形式的单个列名，则返回的对象为经过分组的Series
s_grouped = df.groupby(["key1", "key2"])["data2"]
s_grouped

In [137]:
s_grouped.mean()

key1  key2
a     1       1.008009
      2       0.343925
b     1       1.870438
      2       0.082590
Name: data2, dtype: float64

In [139]:
# 传入列表
s_grouped = df.groupby(["key1", "key2"])[["data1", "data2"]]
s_grouped

In [141]:
s_grouped.mean()

data1     data2
key1 key2                    
a    1     0.037861  1.008009
     2    -1.311733  0.343925
b    1     0.702981  1.870438
     2     0.467942  0.082590

In [147]:
# 利用字典和Series分组
people = pd.DataFrame(np.random.standard_normal((5, 5)),
                      columns=["a", "b", "c", "d", "e"],
                      index=["Joe", "Steve", "Wanda", "Jill", "Trey"])

people.iloc[2:3, [1, 2]] = np.nan   # 第3行，二三列为空
people

,a,b,c,d,e
Joe,1.313307,0.368647,-0.735109,-1.334266,-1.475435
Steve,1.751461,-0.448434,0.592664,0.681826,-0.825526
Wanda,-1.136729,NaN,NaN,0.130935,-1.107013
Jill,0.665139,-1.997900,-0.534056,0.691560,2.544845
Trey,0.000027,-1.162507,-0.302826,0.518771,0.603127


people.iloc[2:3, [1, 2]]：  



2:3表示选择第3行（索引为2），注意这个切片的结束索引不包括在内。  



[1, 2]表示选择第2列（b）和第3列（c）。  

In [150]:
# 假设已知列的分组关系，根据分组计算列的和：
mapping = {"a": "red", "b": "red", "c": "blue",
           "d": "blue", "e": "red", "f": "orange"}

In [154]:
# 将字典传给groupby构造数组，并且包含了数组中不存在的 “f”， 表示即使存在未使用的分组也是可以的
by_column = people.groupby(mapping, axis="columns")
by_column.sum()

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_4456\405393630.py:2: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  by_column = people.groupby(mapping, axis="columns")


,blue,red
Joe,-2.069375,0.206519
Steve,1.274490,0.477501
Wanda,0.130935,-2.243742
Jill,0.157504,1.212085
Trey,0.215945,-0.559354


In [164]:
# 利用转置去计算列的分组关系
by_columns = people.T.groupby(mapping)
print(people.T)
grouped_sum = by_columns.sum()
grouped_sum

        Joe     Steve     Wanda      Jill      Trey
a  1.313307  1.751461 -1.136729  0.665139  0.000027
b  0.368647 -0.448434       NaN -1.997900 -1.162507
c -0.735109  0.592664       NaN -0.534056 -0.302826
d -1.334266  0.681826  0.130935  0.691560  0.518771
e -1.475435 -0.825526 -1.107013  2.544845  0.603127


,Joe,Steve,Wanda,Jill,Trey
blue,-2.069375,1.274490,0.130935,0.157504,0.215945
red,0.206519,0.477501,-2.243742,1.212085,-0.559354


groupby默认按照索引相同的分组，转置后，行变为了abcde,将其替换为mapping对应的str后进行groupby就变味了最后的输出

In [169]:
# 然后再转置回去，和axis=1 效果相同
grouped_sum.T

,blue,red
Joe,-2.069375,0.206519
Steve,1.274490,0.477501
Wanda,0.130935,-2.243742
Jill,0.157504,1.212085
Trey,0.215945,-0.559354


In [181]:
# 简化写法：
people.T.groupby(mapping).sum().T

,blue,red
Joe,-2.069375,0.206519
Steve,1.274490,0.477501
Wanda,0.130935,-2.243742
Jill,0.157504,1.212085
Trey,0.215945,-0.559354


In [171]:
# Series 也有同样的功能，可以看作固定大小的映射
map_series = pd.Series(mapping)
map_series

a       red
b       red
c      blue
d      blue
e       red
f    orange
dtype: object

In [173]:
people.groupby(map_series, axis=1).count()

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_4456\1833935060.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  people.groupby(map_series, axis=1).count()


,blue,red
Joe,2,3
Steve,2,3
Wanda,1,2
Jill,2,3
Trey,2,3


In [177]:
people.T.groupby(map_series).count().T

,blue,red
Joe,2,3
Steve,2,3
Wanda,1,2
Jill,2,3
Trey,2,3


#### 10.1.4 利用函数进行分组  
使用字典和Series外，更经典的是通过py函数定义分组映射的方式  
采用py函数对某行、某列的所有值进行映射  

In [184]:
# 输入人名长度分组
people.groupby(len).sum()

,a,b,c,d,e
3,1.313307,0.368647,-0.735109,-1.334266,-1.475435
4,0.665166,-3.160407,-0.836882,1.210330,3.147972
5,0.614733,-0.448434,0.592664,0.812761,-1.932539


左边的索引就是长度为3、4、5的分组，后面是sum的数值  
例如长度为3的就一个，所以该行的数值不变

In [193]:
# 将函数、数组、列表、字典、Series混合使用也不是问题，因为在内部会被转化为数组：
key_list = ["one", "one", "one", "two", "two"]  # 分组键
people.groupby([len, key_list]).min()  # 分组依据

,,a,b,c,d,e
3,one,1.313307,0.368647,-0.735109,-1.334266,-1.475435
4,two,0.000027,-1.997900,-0.534056,0.518771,0.603127
5,one,-1.136729,-0.448434,0.592664,0.130935,-1.107013


[len, key_list]: 这是分组的依据：  
  
len 是一个函数，表示将 DataFrame 中的每一行的长度（行中元素的数量）作为一个分组键。  
key_list 是一个包含字符串的列表，["one", "one", "one", "two", "two"]，这意味着每个字符串作为另一个分组键。   
这里的分组实际上是将每一行的长度与 key_list 中的值结合起来形成一个复合键进行分组。   

.min(): 对每个分组计算最小值。它会返回每个分组的最小值，通常会应用于数值列。   

**代码解读**  
具体步骤应该是先将key_list作为一列插入到DF中，形成了DF

In [207]:
example = pd.DataFrame(pd.DataFrame(np.random.standard_normal((5, 5)),
                       columns=["a", "b", "c", "d", "e"],
                       index=["Joe", "Steve", "Wanda", "Jill", "Trey"]))
key_list = ["one", "one", "one", "two", "two"]
example.insert(0, "key_list", key_list)
example

,key_list,a,b,c,d,e
Joe,one,0.147085,0.176804,0.469086,-0.015170,-0.514865
Steve,one,-1.686410,0.268944,2.358813,0.237586,-1.641523
Wanda,one,0.468294,0.089392,-0.339201,1.360528,0.769663
Jill,two,0.121702,1.926307,0.930654,0.065188,-0.589080
Trey,two,-0.553684,0.486468,-0.264046,-0.469988,2.625223


In [212]:
# 然后根据len生成新DF
example = pd.DataFrame(pd.DataFrame(np.random.standard_normal((5, 5)),
                       columns=["a", "b", "c", "d", "e"],
                       index=[len(x) for x in ["Joe", "Steve", "Wanda", "Jill", "Trey"]]))
key_list = ["one", "one", "one", "two", "two"]
example.insert(0, "key_list", key_list)
example

,key_list,a,b,c,d,e
3,one,0.296577,0.153305,-1.153766,-0.372565,0.568978
5,one,-0.659195,0.343383,-0.435976,-0.531643,0.226294
5,one,1.000999,0.250195,1.019565,0.322150,-0.727841
4,two,-0.070883,-0.863523,0.465884,0.021202,0.038473
4,two,-1.141285,-0.305171,0.124951,-0.889578,0.655762


In [224]:
# 然后再根据每个组中各列的最小值，并输出
example.groupby(level=0).min()

,key_list,a,b,c,d,e
3,one,0.296577,0.153305,-1.153766,-0.372565,0.568978
4,two,-1.141285,-0.863523,0.124951,-0.889578,0.038473
5,one,-0.659195,0.250195,-0.435976,-0.531643,-0.727841


比如，两个4 two，两行中的每一列，其中第一列最小的值是最后一行的-1.141285，第二列最小的是-0.863523，则输出这个，以此类推

people.groupby([len, key_list]).min()   
这行代码意思就是按照名字长度和所给的key_list作为键进行分组，然后将组内各列的最小值进行输出

#### 10.1.5 根据索引层级分组  
层次化索引数据集最方便的地方就在于它能够根据轴索引的某个层级进行聚合。

In [230]:
# 创建层次化索引
columns = pd.MultiIndex.from_arrays([["US", "US", "US", "JP", "JP"],
                                     [1, 3, 5, 1, 3]],
                                    names=["cty", "tenor"])

hier_df = pd.DataFrame(np.random.standard_normal((4, 5)), columns=columns)
hier_df

cty          US                            JP          
tenor         1         3         5         1         3
0     -0.419931  0.120412 -0.134389 -1.020101 -1.040812
1      0.105837 -1.764879  0.973613 -1.324859  0.072341
2     -0.716938  0.006911  0.495794 -0.188480  0.877146
3     -0.243034  0.087360  0.150759 -0.244813  0.083527

In [232]:
# 层级索引分组，将层级数值或名称传递给level关键字：
hier_df.groupby(level="cty", axis="columns").count()

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_4456\1610983845.py:2: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  hier_df.groupby(level="cty", axis="columns").count()


cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


In [236]:
# 按照 cty 该行的列索引进行分组
hier_df.T.groupby(level="cty").count().T

cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


按照 cty 该行的列索引进行分组，其中就两个，一个US，一个JP，最后在进行求和

### 10.2 数据聚合  



经过优化后的groupby方法：
![jupyter](10.2.png)

也可以自己制定聚合算法，也可以调用已经定义好的任何方法  



Series 中的nsmallest方法可以计算数据中的若干最小值  



虽然nsmallest不是为groupby实现，但仍可以用于groupby对象  



groupby实际上是先对Series进行切片然后再调用nsmallest  

In [248]:
df

,key1,key2,data1,data2
0,a,1,0.037861,1.008009
1,a,2,-1.311733,0.343925
2,None,1,1.226025,-0.321111
3,b,2,0.467942,0.082590
4,b,1,0.702981,1.870438
5,a,<NA>,0.501122,1.692599
6,None,1,0.571569,0.169466


In [252]:
grouped = df.groupby("key1")
grouped["data1"].nsmallest(2)  # 选取每个组合中的最小的两个值

key1   
a     1   -1.311733
      0    0.037861
b     3    0.467942
      4    0.702981
Name: data1, dtype: float64

In [258]:
# 使用自己的聚合函数，只需要将函数传入agg方法：
def peak_to_peak(arr):
    return arr.max() - arr.min()  # 返回每个组中的最大值减去最小值

grouped.agg(peak_to_peak)   # 调用agg

,key2,data1,data2
key1,,,
a,1,1.812854,1.348674
b,1,0.235039,1.787848


In [260]:
# describe也可以用于groupby函数，但它们并非聚合运算：
grouped.describe()

key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0 -0.257583  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0  0.585461  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     0.269491  0.501122   3.0  1.014844  0.674363  0.343925  0.675967   
b     0.644221  0.702981   2.0  0.976514  1.264200  0.082590  0.529552   

                                    
           50%       75%       max  
key1                                
a     1.008009  1.350304  1.692599  
b     0.976514  1.423476  1.870438  

[2 rows x 24 columns]

#### 10.2.1 逐列操作和多函数应用

In [22]:
# 读取csv并追加小费百分比的列
tips = pd.read_csv("../examples/tips.csv")
tips.head()

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


In [24]:
# 追加小费百分比的列
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [26]:
# 对不同的列进行不同的聚合操作：
# 对day和smoker进行分组
grouped = tips.groupby(["day", "smoker"])   # 根据两列一起分组

# 可以将函数以字符串的形式传入：
grouped_pct = grouped["tip_pct"]  # 获取分组后的tip_dic列赋给grouped_pct
grouped_pct.agg("mean")   # 计算每个组合的tip_pct列的平均值

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

grouped 分组和 grouped_pct 分组是两个东西

![jupuyer](10.3.png)

In [28]:
# 如果传入的是列表，则得到的DF的列就是相应的函数名
grouped_pct.agg(["mean", "std", peak_to_peak])  # agg中是小费在总收入的占比，平均、标准差、大小差值

NameError: name 'peak_to_peak' is not defined

In [299]:
# groupby自定义列名：(name, function) 
grouped_pct.agg([("average", "mean"), ("stdev", np.std)])

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_4456\4073686275.py:2: FutureWarning: The provided callable <function std at 0x0000000008A0F920> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  grouped_pct.agg([("average", "mean"), ("stdev", np.std)])


average     stdev
day  smoker                    
Fri  No      0.151650  0.028123
     Yes     0.174783  0.051293
Sat  No      0.158048  0.039767
     Yes     0.147906  0.061375
Sun  No      0.160113  0.042347
     Yes     0.187250  0.154134
Thur No      0.160298  0.038774
     Yes     0.163863  0.039389

In [30]:
# 该警告是说以后agg传递都是以字符串的形式传递，而非数组
grouped_pct.agg([("average", "mean"), ("stdev", "std")])  # 字符串std即代表了数组std

average     stdev
day  smoker                    
Fri  No      0.151650  0.028123
     Yes     0.174783  0.051293
Sat  No      0.158048  0.039767
     Yes     0.147906  0.061375
Sun  No      0.160113  0.042347
     Yes     0.187250  0.154134
Thur No      0.160298  0.038774
     Yes     0.163863  0.039389

In [32]:
# 对于DF，可以定义一个函数列表，应用到所有列，或者不同列对应不同函数
# 对tip_pct 和 total_bill 列计算得到3个相同的统计值：
functions = ["count", "mean", "max"]
result = grouped[["tip_pct", "total_bill"]].agg(functions)
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

如你所见，产生的DataFrame具有层次化的列，这相当于分别对各列进行聚合，然后将列名作为keys参数，并用concat方法将结果拼接到一起：

In [34]:
result["tip_pct"]

count      mean       max
day  smoker                           
Fri  No          4  0.151650  0.187735
     Yes        15  0.174783  0.263480
Sat  No         45  0.158048  0.291990
     Yes        42  0.147906  0.325733
Sun  No         57  0.160113  0.252672
     Yes        19  0.187250  0.710345
Thur No         45  0.160298  0.266312
     Yes        17  0.163863  0.241255

In [36]:
# 与之前一样，这里也可以传入带有自定义名称的元组列表
ftuples = [("Average", "mean"), ("Variance", "var")]  # Variance 方差
grouped[["tip_pct", "total_bill"]].agg(ftuples)

tip_pct           total_bill            
              Average  Variance    Average    Variance
day  smoker                                           
Fri  No      0.151650  0.000791  18.420000   25.596333
     Yes     0.174783  0.002631  16.813333   82.562438
Sat  No      0.158048  0.001581  19.661778   79.908965
     Yes     0.147906  0.003767  21.276667  101.387535
Sun  No      0.160113  0.001793  20.506667   66.099980
     Yes     0.187250  0.023757  24.120000  109.046044
Thur No      0.160298  0.001503  17.113111   59.625081
     Yes     0.163863  0.001551  19.190588   69.808518

In [38]:
# 单列或多列应用不同的函数: agg传入字典，将列名映射到字典：
grouped.agg({"tip": "max", "size": "sum"})  # 分组后的tip列求最大值，size列求和 

tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [40]:
grouped.agg({"tip_pct": ["min", "max", "mean", "std"],
             "size": "sum"})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

**!  只有将多个函数应用到至少一列时，DF才会有层次化索引**

#### 10.2.2 返回不含索引的聚合数据  
对于到目前为止的所有示例，返回的聚合数据都具有索引（可能还是层次化索引），索引由唯一的用于分组的键组合构成。  
由于并不总是需要索引因此你可以向groupby传入as_index=False以禁用该功能：    

In [42]:
tips.groupby(["day", "smoker"], as_index=False).mean(numeric_only=True)

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


### 10.3 Apply：通用的“拆分——应用——联合”范式  
groupby中最常用的就是apply方法，apply会将待处理的对象拆成多个分段，然后对各片段调用传入的函数，最后再将各片拼接到一起

In [49]:
# 根据分组选取五个最高的tip_pct值
# 特定列选取最大值：
def top(df, n=5, columns="tip_pct"):
    return df.sort_values(columns, ascending=False)[:n]  # 传入n=6，对前六个进行切片然后再排序输出每组的最大值

top(tips, n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
232,11.61,3.39,No,Sat,Dinner,2,0.291990
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


降序排序，输出前六行

In [52]:
# 按smoker分组，调用apply以及选取前五个函数，
tips.groupby("smoker").apply(top)

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_1100\759078820.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker").apply(top)


total_bill   tip smoker   day    time  size   tip_pct
smoker                                                           
No     232       11.61  3.39     No   Sat  Dinner     2  0.291990
       149        7.51  2.00     No  Thur   Lunch     2  0.266312
       51        10.29  2.60     No   Sun  Dinner     2  0.252672
       185       20.69  5.00     No   Sun  Dinner     5  0.241663
       88        24.71  5.85     No  Thur   Lunch     2  0.236746
Yes    172        7.25  5.15    Yes   Sun  Dinner     2  0.710345
       178        9.60  4.00    Yes   Sun  Dinner     2  0.416667
       67         3.07  1.00    Yes   Sat  Dinner     1  0.325733
       183       23.17  6.50    Yes   Sun  Dinner     4  0.280535
       109       14.31  4.00    Yes   Sat  Dinner     2  0.279525

这个警告是因为你在使用 groupby 后的 apply 函数时，没有明确处理分组列。为了消除警告，你可以在调用 apply 时，传递 include_groups=False，或者在 apply 之前选择要处理的列。

In [55]:
tips.groupby("smoker").apply(top, include_groups=False)

total_bill   tip   day    time  size   tip_pct
smoker                                                    
No     232       11.61  3.39   Sat  Dinner     2  0.291990
       149        7.51  2.00  Thur   Lunch     2  0.266312
       51        10.29  2.60   Sun  Dinner     2  0.252672
       185       20.69  5.00   Sun  Dinner     5  0.241663
       88        24.71  5.85  Thur   Lunch     2  0.236746
Yes    172        7.25  5.15   Sun  Dinner     2  0.710345
       178        9.60  4.00   Sun  Dinner     2  0.416667
       67         3.07  1.00   Sat  Dinner     1  0.325733
       183       23.17  6.50   Sun  Dinner     4  0.280535
       109       14.31  4.00   Sat  Dinner     2  0.279525

基于smoker的值，将tips分组，在各个分组上调用函数top，由pd进行拼接并使用分组名作为各组的标签，最后就有了一个层次化索引，内层索引包含原DF的索引值。

In [58]:
# 如果让传给apply的函数还接收其他参数或关键字，可以将这些内容放在函数名后面一并传入：
tips.groupby(["smoker", "day"]).apply(top, n=1, columns="total_bill", include_groups=False)

total_bill    tip    time  size   tip_pct
smoker day                                                
No     Fri  94        22.75   3.25  Dinner     2  0.142857
       Sat  212       48.33   9.00  Dinner     4  0.186220
       Sun  156       48.17   5.00  Dinner     6  0.103799
       Thur 142       41.19   5.00   Lunch     5  0.121389
Yes    Fri  95        40.17   4.73  Dinner     4  0.117750
       Sat  170       50.81  10.00  Dinner     3  0.196812
       Sun  182       45.35   3.50  Dinner     3  0.077178
       Thur 197       43.11   5.00   Lunch     4  0.115982

将"smoker", "day"两列进行组合，经过top函数排序后，每组选取total_bill第一个最大的那个行，并且不处理分组列

In [64]:
# describe用法
result = tips.groupby("smoker")["tip_pct"].describe()
result

,count,mean,std,min,25%,50%,75%,max
smoker,,,,,,,,
No,151.0,0.159328,0.039910,0.056797,0.136906,0.155625,0.185014,0.291990
Yes,93.0,0.163196,0.085119,0.035638,0.106771,0.153846,0.195059,0.710345


In [70]:
result.unstack("smoker")  # 将smoker列索引透视为行索引，因为之前有行索引，所以直接就是行标签

       smoker
count  No        151.000000
       Yes        93.000000
mean   No          0.159328
       Yes         0.163196
std    No          0.039910
       Yes         0.085119
min    No          0.056797
       Yes         0.035638
25%    No          0.136906
       Yes         0.106771
50%    No          0.155625
       Yes         0.153846
75%    No          0.185014
       Yes         0.195059
max    No          0.291990
       Yes         0.710345
dtype: float64

In [74]:
# 在groupby内部，当调用describe之类的方法时，实际上是执行了以下代码：
def f(group):
    return group.describe()

grouped.apply(f, include_groups=False)

total_bill       tip  size   tip_pct
day  smoker                                            
Fri  No     count    4.000000  4.000000  4.00  4.000000
            mean    18.420000  2.812500  2.25  0.151650
            std      5.059282  0.898494  0.50  0.028123
            min     12.460000  1.500000  2.00  0.120385
            25%     15.100000  2.625000  2.00  0.137239
...                       ...       ...   ...       ...
Thur Yes    min     10.340000  2.000000  2.00  0.090014
            25%     13.510000  2.000000  2.00  0.148038
            50%     16.470000  2.560000  2.00  0.153846
            75%     19.810000  4.000000  2.00  0.194837
            max     43.110000  5.000000  4.00  0.241255

[64 rows x 4 columns]

#### 10.3.1 禁用分组键  
从上面的例子中可以看出，分组键会与原始对象各分块的索引共同构成结果对象中的层次化索引。

In [77]:
# 将group_keys=False传入groupby即可禁止该效果：
tips.groupby("smoker", group_keys=False).apply(top, include_groups=False)

,total_bill,tip,day,time,size,tip_pct
232,11.61,3.39,Sat,Dinner,2,0.291990
149,7.51,2.00,Thur,Lunch,2,0.266312
51,10.29,2.60,Sun,Dinner,2,0.252672
185,20.69,5.00,Sun,Dinner,5,0.241663
88,24.71,5.85,Thur,Lunch,2,0.236746
172,7.25,5.15,Sun,Dinner,2,0.710345
178,9.60,4.00,Sun,Dinner,2,0.416667
67,3.07,1.00,Sat,Dinner,1,0.325733
183,23.17,6.50,Sun,Dinner,4,0.280535
109,14.31,4.00,Sat,Dinner,2,0.279525


group_keys=False 的作用是，结果中不会包含 "smoker" 这一分组键作为额外的索引层级。  



![jupyter](10.4.png)

#### 10.3.2 分位数和桶分析  
pd.cut 与 pd.qcut 结合groupby轻松实现数据集的桶分析和分位数分析。

In [82]:
# 使用pd.cut实现等长桶分类：
frame = pd.DataFrame({"data1": np.random.standard_normal(1000),
                      "data2": np.random.standard_normal(1000)})

frame.head()

,data1,data2
0,-1.560352,0.573449
1,1.789054,-2.515551
2,-0.357584,1.407893
3,-1.110666,0.039728
4,1.438244,0.312885


In [86]:
quartiles = pd.cut(frame["data1"], 4)  # 将data1分为四个等长的区间
quartiles.head(10)

0    (-2.343, -0.491]
1      (1.361, 3.213]
2     (-0.491, 1.361]
3    (-2.343, -0.491]
4      (1.361, 3.213]
5     (-0.491, 1.361]
6     (-0.491, 1.361]
7    (-2.343, -0.491]
8    (-2.343, -0.491]
9     (-0.491, 1.361]
Name: data1, dtype: category
Categories (4, interval[float64, right]): [(-4.203, -2.343] < (-2.343, -0.491] < (-0.491, 1.361] < (1.361, 3.213]]

**由cut返回的Categories对象可以直接传递给groupby**

In [91]:
# 计算分位数的分组统计集合：
def get_stats(group):
    return pd.DataFrame(
        {"min": group.min(), "max": group.max(),
         "count": group.count(), "mean": group.mean()}
    )

grouped = frame.groupby(quartiles)
grouped.apply(get_stats, include_groups=False)

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_1100\1122030852.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = frame.groupby(quartiles)


min       max  count      mean
data1                                                      
(-4.203, -2.343] data1 -4.195111 -2.438793     11 -3.012277
                 data2 -1.265911  1.310153     11  0.005763
(-2.343, -0.491] data1 -2.281331 -0.494922    313 -1.103445
                 data2 -2.281685  3.007311    313  0.020045
(-0.491, 1.361]  data1 -0.483084  1.345395    594  0.326676
                 data2 -3.118264  2.826408    594 -0.018495
(1.361, 3.213]   data1  1.362175  3.212932     82  1.826774
                 data2 -2.611250  2.028969     82 -0.004943

这个警告是因为在使用 groupby 时，默认参数 observed 的值将在未来版本的 Pandas 中改变。当前默认为 False，意味着所有分组都会被考虑，而在未来版本中将改为 True，仅考虑实际存在的分组。

In [104]:
# 传入特定的组，输出每个组的最大、最小、平均、累加值
def get_stats(group):
    return pd.DataFrame(
        {"min": group.min(), "max": group.max(),
         "count": group.count(), "mean": group.mean()}
    )

# 将frame按照qua划分的区域分组，每组都有两个行索引，data1和data2
grouped = frame.groupby(quartiles, observed=False)

# 将函数用于每个分组，返回统计的信息，不包含分组键
grouped.apply(get_stats, include_groups=False)

min       max  count      mean
data1                                                      
(-4.203, -2.343] data1 -4.195111 -2.438793     11 -3.012277
                 data2 -1.265911  1.310153     11  0.005763
(-2.343, -0.491] data1 -2.281331 -0.494922    313 -1.103445
                 data2 -2.281685  3.007311    313  0.020045
(-0.491, 1.361]  data1 -0.483084  1.345395    594  0.326676
                 data2 -3.118264  2.826408    594 -0.018495
(1.361, 3.213]   data1  1.362175  3.212932     82  1.826774
                 data2 -2.611250  2.028969     82 -0.004943

该函数的分组键是quartiles 和 data1、data2 这段区域不会被执行

In [109]:
# 以上的所有操作都可以用一句话执行
grouped.agg(["min", "max", "count", "mean"])

data1                               data2            \
                       min       max count      mean       min       max   
data1                                                                      
(-4.203, -2.343] -4.195111 -2.438793    11 -3.012277 -1.265911  1.310153   
(-2.343, -0.491] -2.281331 -0.494922   313 -1.103445 -2.281685  3.007311   
(-0.491, 1.361]  -0.483084  1.345395   594  0.326676 -3.118264  2.826408   
(1.361, 3.213]    1.362175  3.212932    82  1.826774 -2.611250  2.028969   

                                  
                 count      mean  
data1                             
(-4.203, -2.343]    11  0.005763  
(-2.343, -0.491]   313  0.020045  
(-0.491, 1.361]    594 -0.018495  
(1.361, 3.213]      82 -0.004943

In [111]:
# 等频桶：
quartiles_samp = pd.qcut(frame["data1"], 4, labels=False)  # labels=False获取分位数的索引
quartiles_samp.head()

0    0
1    3
2    1
3    0
4    3
Name: data1, dtype: int64

使用 pd.qcut() 或 pd.cut()：这些函数用于将数据分为多个区间（bins）。  




labels=False：如果设置为 True，输出将是每个区间的标签（区间的名称）。而如果设置为 False，输出将是每个数据点所属区间的索引值（即该数据点属于哪个区间，通常是整数值）。

In [115]:
# 切片带入分组
grouped = frame.groupby(quartiles_samp)
grouped.apply(get_stats)

min       max  count      mean
data1                                           
0     data1 -4.195111 -0.709489    250 -1.335767
      data2 -2.281685  3.007311    250 -0.046796
1     data1 -0.708789 -0.026428    250 -0.355708
      data2 -3.118264  2.918157    250  0.052501
2     data1 -0.018803  0.628156    250  0.307561
      data2 -2.751305  2.453658    250  0.072326
3     data1  0.628707  3.212932    250  1.245225
      data2 -2.611250  2.460480    250 -0.098247

#### 10.3.3 实战：用指定分组的填充值填充空缺值：
在清洗数据时，有时用dropna将其行删除，但有时可能想用固定值或者数据本身衍生出的值来填充空缺Na，这时候可以使用fillna工具

In [120]:
# 平均值填充空值：
s = pd.Series(np.random.standard_normal(6))
s[::2] = np.nan
s

0         NaN
1   -0.826050
2         NaN
3   -1.755640
4         NaN
5    1.372007
dtype: float64

s[::2]  **将所有偶数索引位置填入Na**



切片语法的格式为 start:stop:step   



start（起始索引）：省略时默认从 0 开始。  
stop（结束索引）：省略时默认到 Series 的末尾。  
step（步长）：这里设置为 2，表示每次选择一个元素后跳过一个元素。  



s[::2] 实际上从索引 0 开始，依次选择偶数索引位置的元素（0、2、4、...），直到 Series 的末尾。

In [126]:
s.fillna(s.mean())

0   -0.403228
1   -0.826050
2   -0.403228
3   -1.755640
4   -0.403228
5    1.372007
dtype: float64

**对不同的分组填充不同的值**  




**方法1：将数据分组，并使用apply和一个能够对数据块调用的fillna函数。**

In [135]:
states = ["Ohio", "New York", "Vermont", "Florida",
          "Oregon", "Nevada", "California", "Idaho"]

group_key = ["East", "East", "East", "East",
             "West", "West", "West", "West"]

data = pd.Series(np.random.standard_normal(8), index=states)
data

Ohio         -0.614482
New York     -1.678150
Vermont      -0.290824
Florida       0.560931
Oregon        0.484076
Nevada        2.737876
California   -1.525715
Idaho        -0.159014
dtype: float64

In [137]:
# 将其中一些设置为缺失值：
data[["Vermont", "Nevada", "Idaho"]] = np.nan
data

Ohio         -0.614482
New York     -1.678150
Vermont            NaN
Florida       0.560931
Oregon        0.484076
Nevada             NaN
California   -1.525715
Idaho              NaN
dtype: float64

In [139]:
data.groupby(group_key).size()

East    4
West    4
dtype: int64

In [141]:
data.groupby(group_key).count()

East    3
West    2
dtype: int64

In [143]:
data.groupby(group_key).mean()

East   -0.577233
West   -0.520819
dtype: float64

count() 统计数量，而不处理数值的总和。  



mean() 则计算数值的平均。


count() 不是对值进行相加，而是统计每组的元素数量，而 mean() 则是对组内的数值进行求平均。

In [146]:
# 用分组平均值对Na进行填充：
def fill_mean(group):
    return group.fillna(group.mean())

data.groupby(group_key).apply(fill_mean)

East  Ohio         -0.614482
      New York     -1.678150
      Vermont      -0.577233
      Florida       0.560931
West  Oregon        0.484076
      Nevada       -0.520819
      California   -1.525715
      Idaho        -0.520819
dtype: float64

所以此段代码就是对East组用East单有的平均值填充，而West用其单有的平均值填充

In [160]:
# 利用预设的填充值：
fill_values = {"East": 0.5, "West": -1}

def fill_func(group):
    return group.fillna(fill_values[group.name])   # 将 NaN 值替换为当前组名对应的填充值

# 对每个分组应用 fill_func 函数，填充 NaN 值
data.groupby(group_key).apply(fill_func)

East  Ohio         -0.614482
      New York     -1.678150
      Vermont       0.500000
      Florida       0.560931
West  Oregon        0.484076
      Nevada       -1.000000
      California   -1.525715
      Idaho        -1.000000
dtype: float64

#### 10.3.4 实战：随机采样和排列    
假设你想要从一个大型数据集中随机抽取（进行替换或不替换）样本，以进行蒙特卡罗模拟（MonteCarlosimulation）或其他工作。“抽取”的方式有很多，这里采用Series的sample方法。

In [169]:
# 创建一副英式扑克牌：
suits = ["H", "S", "C", "D"]  # 红桃、黑桃、梅花、方块
card_val = (list(range(1, 11)) + [10] * 3) * 4   # 生成从1到10的数字，并且再生成3个10，共4个10
base_names = ["A"] + list(range(2, 11)) + ["J", "K", "Q"]  # 牌面
cards = []
for suit in suits:
    cards.extend(str(num) + suit for num in base_names)  # 将牌面和花色组合添加到cards中

deck = pd.Series(card_val, index=cards)  #创建Series，其中 card_val 作为值，cards 作为索引。

现在就有了一个长度等于52的Series，其索引包括牌名和牌值，其中的值则是用于21点或其他游戏中用于计分的点数（为了简单起见，令"A"的点数为1）：

In [167]:
deck.head(13)

AH      1
2H      2
3H      3
4H      4
5H      5
6H      6
7H      7
8H      8
9H      9
10H    10
JH     10
KH     10
QH     10
dtype: int64

str(num) + suit for num in base_names 是两个操作，str + x,  x 是遍历的base_names中的牌面  
suit 共四个花色，牌面13个，遍历下来一共 4 * 13 = 52个

[10] * 3  #  结果是 [10, 10, 10]  而不是位置10上的值出现三次

extend 方法是将一个可迭代对象（如列表）中的元素追加到现有列表的末尾。  
append 只能追加一个  

In [188]:
# 随机抽取 5 张：
def draw(deck, n=5):
    return deck.sample(n)   # sample从deck中随机抽取5个样本

draw(deck)

AD     1
3S     3
2H     2
QC    10
5S     5
dtype: int64

sample 函数用于从 DataFrame 或 Series 中随机抽取样本。它允许你指定抽取的数量、抽样的方式以及是否允许重复等参数。

In [199]:
# 每种花色随机抽取两个：
# 由于花色是牌面的最后一个字符，可以用apply进行分组：
def get_suit(card):
    # 最后一个字母是花色
    return card[-1]

deck.groupby(get_suit).apply(draw, n=2)

C  JC    10
   4C     4
D  3D     3
   JD    10
H  8H     8
   2H     2
S  AS     1
   7S     7
dtype: int64

deck.groupby(get_suit) 将deck中的牌以花色分组，  
apply(draw, n=2)将分好的组每组中随机抽取两张牌



第一步时，每个花色对应13张牌，然后第二步从13张中抽两张

**为什么get_suit是一个函数，却能作为分组依据？**  



因为 groupby 方法可以接受一个函数作为分组依据。这是通过在每个数据点上调用该函数来实现的。



1.函数作为分组依据：  

当你传递一个函数（如 get_suit）给 groupby 时，Pandas 会对 deck 中的每个元素调用这个函数。  
函数的返回值将作为分组的键。也就是说，每张牌的花色将用于确定它所属的组。  




2.get_suit 的工作方式：  

get_suit(card) 函数从每张牌的字符串中提取最后一个字符（表示花色），然后返回该花色。  
例如，对于牌 "3H"（红桃），get_suit("3H") 将返回 "H"。  

In [209]:
# 也可以传入 group_keys=False，丢弃外层的花色索引，只保留选取的牌：
# 按花色分组不保留花色索引标签
deck.groupby(get_suit, group_keys=False).apply(draw, n=2)

KC    10
2C     2
6D     6
2D     2
JH    10
8H     8
6S     6
5S     5
dtype: int64

#### 实战：分组加权平均和相关系数  
根据“拆分”——“应用”——“联合”范式，进行DF的列与列和Series之间的运算，如加权平均：

In [214]:
# 分组键、值、权重
df = pd.DataFrame({"category": ["a", "a", "a", "a",
                                "b", "b", "b", "b"],
                   "data": np.random.standard_normal(8),
                   "weights": np.random.uniform(size=8)})
df

,category,data,weights
0,a,-0.670752,0.577387
1,a,0.185882,0.536407
2,a,0.202940,0.532995
3,a,0.511434,0.987936
4,b,1.462311,0.388052
5,b,-0.409442,0.014949
6,b,-0.441377,0.032550
7,b,0.733918,0.763248


np.average(a, axis, weights, returned),np中average本身就有计算权重的可选参数  




a：要计算平均值的数组或序列。  

axis（可选）：指定计算平均值的轴。默认是 None，表示对整个数组计算平均值。  

weights（可选）：与 a 相同形状的数组，指定每个元素的权重。如果指定，则计算加权平均值。  

returned（可选）：布尔值，默认为 False。如果设置为 True，则返回一个元组，包含计算出的平均值和权重的总和。  

In [232]:
# 利用categorie 计算加权平均值：
grouped = df.groupby("category")   # 按category列分组

def get_wavg(group):
    return np.average(group["data"], weights=group["weights"])  # 对每组计算 data 列的加权平均值，使用 weights 列作为权重

grouped.apply(get_wavg, include_groups=False)  # 对每个分组应用函数，计算每个类别的加权平均值

category
a    0.123677
b    0.923529
dtype: float64

np.average(group["data"], weights=group["weights"])  是group不是groupby，二者不同，这句代码旨在计算group["data"] 列的加权平均值  



1.group["data"]：  

这是当前分组中的 data 列的值。  




2.weights=group["weights"]：  

这是当前分组中的 weights 列的值，用作加权。 




3.np.average()：  
 
这是 NumPy 提供的函数，用于计算数组的平均值。通过指定 weights 参数，可以计算加权平均值。  

**加权平均值：通过将每个数值乘以其对应的权重，然后求和，再除以所有权重的总和。**  


![jupyter](10.5.png)
![jupyter](10.6.png)

In [236]:
# 股票实例
#  parse_dates=True 自动解析日期列，自动转为datetime         index_col=0第一列作为索引
close_px = pd.read_csv("../examples/stock_px.csv", parse_dates=True, index_col=0)
close_px.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2214 entries, 2003-01-02 to 2011-10-14
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2214 non-null   float64
 1   MSFT    2214 non-null   float64
 2   XOM     2214 non-null   float64
 3   SPX     2214 non-null   float64
dtypes: float64(4)
memory usage: 86.5 KB


info() 方法可以快速了解 DataFrame 的全貌结构，检查是否存在缺失值，以及各列的数据类型，这在数据清理和预处理阶段特别有用。

计算由日收益率（通过百分数变化计算）与SPX之间的年化相关系数组成的DataFrame，  
创建一个函数，用它计算每列和"SPX"列对应的相关系数：  

In [247]:
def spx_corr(group):
    return group.corrwith(group["SPX"])  # corrwith相关性计算，计算输入的组与SPX的相关性

# 使用pct_change计算close_px的百分比变化:
rets = close_px.pct_change().dropna()  # pct_change是计算当前行与前一行之间的百分比变化

# 按照年份将百分比变化进行分组，可以用一个单行函数返回每个datetime标签的year属性，这样可以从每行标签提取年份
def get_year(x):
    return x.year  # 获取日期的年份部分

by_year = rets.groupby(get_year)   # 根据年份对rets进行分组
by_year.apply(spx_corr)  # 对之前按年份分的组调用spx_corr函数，即计算每个组内（年份内）各列与SPX的相关性

,AAPL,MSFT,XOM,SPX
2003,0.541124,0.745174,0.661265,1.0
2004,0.374283,0.588531,0.557742,1.0
2005,0.467540,0.562374,0.631010,1.0
2006,0.428267,0.406126,0.518514,1.0
2007,0.508118,0.658770,0.786264,1.0
2008,0.681434,0.804626,0.828303,1.0
2009,0.707103,0.654902,0.797921,1.0
2010,0.710105,0.730118,0.839057,1.0
2011,0.691931,0.800996,0.859975,1.0


In [251]:
# 计算列与列之间的相关系数，
# 计算Apple和MSFT微软的年度相关系数
def corr_aapl_msft(group):
    return group["AAPL"].corr(group["MSFT"])  # corr计算相关系数

by_year.apply(corr_aapl_msft)

2003    0.480868
2004    0.259024
2005    0.300093
2006    0.161735
2007    0.417738
2008    0.611901
2009    0.432738
2010    0.571946
2011    0.581987
dtype: float64

#### 10.3.6 实战：分组线性回归  
regress函数利用stats models(计量经济学库)对各数据块执行普通最小二乘回归：

In [258]:
import statsmodels.api as sm

def regress(data, yvar=None, xvars=None):
    Y = data[yvar]
    X = data[xvars]
    X["intercept"] = 1
    result = sm.OLS(Y, X).fit()
    return result.params

In [260]:
# 计算AAPL对SPX收益率的年化线性回归：
by_year.apply(regress, yvar="AAPL", xvars=["SPX"])

,SPX,intercept
2003,1.195406,0.000710
2004,1.363463,0.004201
2005,1.766415,0.003246
2006,1.645496,0.000080
2007,1.198761,0.003438
2008,0.968016,-0.001110
2009,0.879103,0.002954
2010,1.052608,0.001261
2011,0.806605,0.001514


### 10.4 分组转换和“展开式”Groupby 运算  
内置的transform方法，类似于apply，可以对使用的函数有更多的限制：  


1.生成标量值，能传播到分组形状  
2.生成与输入分组具有相同形状  
3.无法修改输入  

In [285]:
df = pd.DataFrame({"key": ["a", "b", "c"] * 4,
                   "value": np.arange(12.)})
df

,key,value
0,a,0.0
1,b,1.0
2,c,2.0
3,a,3.0
4,b,4.0
5,c,5.0
6,a,6.0
7,b,7.0
8,c,8.0
9,a,9.0


In [269]:
# 按照键计算分组平均值
g = df.groupby("key")["value"]  # 按照key计算value的平均值
g.mean()

key
a    4.5
b    5.5
c    6.5
Name: value, dtype: float64

如果我们想要创建一个Series，它的形状与df["value"]相同，但值替换为“key”的分组平均值，我们可以向transform传入一个计算单个分组平均值的函数

In [278]:
def get_mean(group):
    return group.mean()   # 接收组并返回该组的平均值

g.transform(get_mean)  # transform(get_mean) 方法会对 g 中的每个组应用 get_mean 函数。

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

人话： 就是将分过组的数通过transform赋值给每个组  
a 求出平均值是 4.5，那么 a 组内全为4.5，以此类推

与 apply 方法不同，transform 会返回一个与原始 DataFrame 或 Series 形状相同的对象，每个元素被替换为其所属组的平均值。

transform 是 Pandas 中的一个方法，用于对分组对象（如通过 groupby 创建的对象）应用函数，并返回一个与原始数据形状相同的结果。

In [294]:
# 对于内置的聚合函数，可以像Groupby的agg方法那样，传入函数的字符串名：
g.transform("mean")   # 与前面的写法效果相同

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

In [300]:
# 类似于apply，transform可以使用能返回Series的函数，但结果未必与输入具有相同的大小。
# 利用辅助函数将各分组乘2
def times_two(group):
    return group * 2    # 将组中的数值乘 2，而不是元素个数翻倍

g.transform(times_two)

0      0.0
1      2.0
2      4.0
3      6.0
4      8.0
5     10.0
6     12.0
7     14.0
8     16.0
9     18.0
10    20.0
11    22.0
Name: value, dtype: float64

类似于apply，transform可以使用能返回Series的函数，但结果未必与输入具有相同的大小。  
人话：  


apply 可以产生不同大小的结果，适用于更复杂的操作和聚合。  
transform 强调保持输入和输出的形状一致，适用于需要逐元素转换的情况。 

它们的输出形状和大小的处理方式是不同的。

In [308]:
# 计算各分组的降序排名：
def get_ranks(group):
    return group.rank(ascending=False)

g.transform(get_ranks)

0     4.0
1     4.0
2     4.0
3     3.0
4     3.0
5     3.0
6     2.0
7     2.0
8     2.0
9     1.0
10    1.0
11    1.0
Name: value, dtype: float64

In [315]:
# 组中数据的标准化处理
def normalize(x):
    return (x - x.mean()) / x.std()   # 偏差除以标准差，从而得到标准化后的值。

# 对于这种情况，transform和apply效果相同
g.transform(normalize)

0    -1.161895
1    -1.161895
2    -1.161895
3    -0.387298
4    -0.387298
5    -0.387298
6     0.387298
7     0.387298
8     0.387298
9     1.161895
10    1.161895
11    1.161895
Name: value, dtype: float64

**为什么 transform 和 apply 效果相同**  
在这种情况下，transform 和 apply 结果相同，因为 normalize 函数的返回值大小与输入相同（即每个组的长度保持不变）。如果 normalize 进行了聚合（如返回单个值），那么 apply 的结果会与 transform 不同。因此，对于某些函数，使用 transform 更加高效且语义明确。

In [319]:
g.apply(normalize)

key    
a    0    -1.161895
     3    -0.387298
     6     0.387298
     9     1.161895
b    1    -1.161895
     4    -0.387298
     7     0.387298
     10    1.161895
c    2    -1.161895
     5    -0.387298
     8     0.387298
     11    1.161895
Name: value, dtype: float64

"mean" 或 "sum"等内置函数通常比一般的apply函数快的多，这些函数在配合transform使用时也存在告诉路径。

In [322]:
# 展开式分组运算：
g.transform("mean")

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

In [324]:
normalized = (df["value"] - g.transform("mean")) / g.transform("std")
normalized

0    -1.161895
1    -1.161895
2    -1.161895
3    -0.387298
4    -0.387298
5    -0.387298
6     0.387298
7     0.387298
8     0.387298
9     1.161895
10    1.161895
11    1.161895
Name: value, dtype: float64

这里，我们直接在多个GroupBy运算输出的结果之间进行算数运算，  
而非编写一个函数，再将其传给groupby（...）.apply。这就是“展开式”要表达的含义。  

### 10.5 透视表和交叉表  
透视表是各种电子表格程序和其他数据分析软件中常见的数据汇总工具。它根据一个或多个键对数据进行聚合，并根据行和列上的分组将数据分配到矩形区域中。


在pd中，结合groupby和层次化索引重塑进行制作透视表。



pd拥有一个顶级的函数，pivot_table函数，除了为groupby提供便利的接口，pivot_table函数还可以添加分项汇总，也叫做差额。

In [329]:
# 计算分组平均数，pivot_table的默认聚合类型，并在行方向上根据day和smoker排列：
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [345]:
tips.pivot_table(index=["day", "smoker"], values=["total_bill", "tip", "size", "tip_pct"], aggfunc='mean')

size       tip   tip_pct  total_bill
day  smoker                                          
Fri  No      2.250000  2.812500  0.151650   18.420000
     Yes     2.066667  2.714000  0.174783   16.813333
Sat  No      2.555556  3.102889  0.158048   19.661778
     Yes     2.476190  2.875476  0.147906   21.276667
Sun  No      2.929825  3.167895  0.160113   20.506667
     Yes     2.578947  3.516842  0.187250   24.120000
Thur No      2.488889  2.673778  0.160298   17.113111
     Yes     2.352941  3.030000  0.163863   19.190588

**与书上不同，不传入足够的参数，pivot_table不会工作**  
书中只需要聚合的列名，就可以直接执行默认操作，现在少一个都不行。

In [349]:
# 通过groupby实现：
tips.groupby(["day", "smoker"]).mean(numeric_only=True)

total_bill       tip      size   tip_pct
day  smoker                                          
Fri  No       18.420000  2.812500  2.250000  0.151650
     Yes      16.813333  2.714000  2.066667  0.174783
Sat  No       19.661778  3.102889  2.555556  0.158048
     Yes      21.276667  2.875476  2.476190  0.147906
Sun  No       20.506667  3.167895  2.929825  0.160113
     Yes      24.120000  3.516842  2.578947  0.187250
Thur No       17.113111  2.673778  2.488889  0.160298
     Yes      19.190588  3.030000  2.352941  0.163863

In [351]:
# 只对tip_pct和size求平局值，根据time分组，把somker放到列上，time和day放在行上：
tips.pivot_table(index=["time", "day"], columns="smoker",
                 values=["tip_pct", "size"])

size             tip_pct          
smoker             No       Yes        No       Yes
time   day                                         
Dinner Fri   2.000000  2.222222  0.139622  0.165347
       Sat   2.555556  2.476190  0.158048  0.147906
       Sun   2.929825  2.578947  0.160113  0.187250
       Thur  2.000000       NaN  0.159744       NaN
Lunch  Fri   3.000000  1.833333  0.187735  0.188937
       Thur  2.500000  2.352941  0.160311  0.163863

将["time", "day"]组合作为行索引，将 smoker 列作为列索引，根据 smoker 的值（如 "Yes" 或 "No"）进行分类。  
values=["tip_pct", "size"]: 指定要聚合的值列。在这里，聚合的列是 tip_pct 和 size。


行索引是time和day组成，列索引是吸烟者的身份，因为是size和tip_pct的值，所以对其尽行聚合后就可以得到吸烟者在固定时间和日期的规模和给的小费占比

In [355]:
# 通过margins=True添加分项汇总，可以对表进行扩充。
# 并且会添加标签为ALL的行和列，对应的值是单行或单列中所有数据的分组统计值
tips.pivot_table(index=["time", "day"], columns="smoker",
                 values=["tip_pct", "size"], margins=True)

size                       tip_pct                    
smoker             No       Yes       All        No       Yes       All
time   day                                                             
Dinner Fri   2.000000  2.222222  2.166667  0.139622  0.165347  0.158916
       Sat   2.555556  2.476190  2.517241  0.158048  0.147906  0.153152
       Sun   2.929825  2.578947  2.842105  0.160113  0.187250  0.166897
       Thur  2.000000       NaN  2.000000  0.159744       NaN  0.159744
Lunch  Fri   3.000000  1.833333  2.000000  0.187735  0.188937  0.188765
       Thur  2.500000  2.352941  2.459016  0.160311  0.163863  0.161301
All          2.668874  2.408602  2.569672  0.159328  0.163196  0.160803

这里，ALL的值为平均值，没有考虑吸烟者与非吸烟者（ALL列)，也没有考虑行上（ALL行）的任意两个分组层级。

In [360]:
# 使用mean以外的其他聚合函数，可以将其传给aggfunc关键字参数
# 使用count和len得到有关分组大小的交叉表（计数或频率）
tips.pivot_table(index=["time", "smoker"], columns="day",
                 values="tip_pct", aggfunc=len, margins=True)

day             Fri   Sat   Sun  Thur  All
time   smoker                             
Dinner No       3.0  45.0  57.0   1.0  106
       Yes      9.0  42.0  19.0   NaN   70
Lunch  No       1.0   NaN   NaN  44.0   45
       Yes      6.0   NaN   NaN  17.0   23
All            19.0  87.0  76.0  62.0  244

"count" 会排除分组数据计数中的空值，而 len 不会

index=["time", "smoker"]: 将 time 和 smoker 列作为行索引。这意味着透视表将根据这些列的组合进行分组，显示不同时间（如 "Lunch" 和 "Dinner"）和吸烟状态（如 "Yes" 或 "No"）的组合。  

columns="day": 将 day 列作为列索引。这意味着透视表的列将根据 day 的值（如 "Sat" 和 "Sun"）进行分类。    

values="tip_pct": 指定要聚合的值列。在这里，聚合的列是 tip_pct。     

aggfunc=len: 使用 len 函数作为聚合函数，表示统计每个组合下的记录数（即在每个时间和吸烟状态的组合中，针对每个星期几的小费百分比的记录数）。    

margins=True: 添加总计行和列，这样可以在表的右侧和底部看到每一列和每一行的总计。    


In [366]:
# 某些组合值为空Na，可以传入fill_value填充：
tips.pivot_table(index=["time", "smoker", "size"], columns="day",
                 values="tip_pct", fill_value=0)

day                      Fri       Sat       Sun      Thur
time   smoker size                                        
Dinner No     1     0.000000  0.137931  0.000000  0.000000
              2     0.139622  0.162705  0.168859  0.159744
              3     0.000000  0.154661  0.152663  0.000000
              4     0.000000  0.150096  0.148143  0.000000
              5     0.000000  0.000000  0.206928  0.000000
              6     0.000000  0.000000  0.103799  0.000000
       Yes    1     0.000000  0.325733  0.000000  0.000000
              2     0.171297  0.148668  0.207893  0.000000
              3     0.000000  0.144995  0.152660  0.000000
              4     0.117750  0.124515  0.193370  0.000000
              5     0.000000  0.106572  0.065660  0.000000
Lunch  No     1     0.000000  0.000000  0.000000  0.181728
              2     0.000000  0.000000  0.000000  0.166005
              3     0.187735  0.000000  0.000000  0.084246
              4     0.000000  0.000000  0.000000  0.138919
              5     0.000000  0.000000  0.000000  0.121389
              6     0.000000  0.000000  0.000000  0.173706
       Yes    1     0.223776  0.000000  0.000000  0.000000
              2     0.181969  0.000000  0.000000  0.158843
              3     0.000000  0.000000  0.000000  0.204952
              4     0.000000  0.000000  0.000000  0.155410

pivot_table 选项：

![jupyter](10.7.png)

#### 交叉表  
交叉表（cross-tabulation，简称crosstab）是一种用于计算分组频次的特殊透视表。

In [378]:
from io import StringIO

data ="""Sample Nationality Handedness
1 USA Right-handed
2 Japan Left-handed
3 USA Right-handed
4 Japan Right-handed
5 Japan Left-handed
6 Japan Right-handed
7 USA Right-handed
8 USA Left-handed
9 Japan Right-handed
10 USA Right-handed"""

# sep=r"\s+" 正则表达式将空格字符、连续的空格字符制表符当作分隔符  r表示原始字符串，告诉py这不是转义字符
data = pd.read_table(StringIO(data), sep=r"\s+")
data

,Sample,Nationality,Handedness
0,1,USA,Right-handed
1,2,Japan,Left-handed
2,3,USA,Right-handed
3,4,Japan,Right-handed
4,5,Japan,Left-handed
5,6,Japan,Right-handed
6,7,USA,Right-handed
7,8,USA,Left-handed
8,9,Japan,Right-handed
9,10,USA,Right-handed


In [386]:
# 根据国籍调查惯用手
pd.pivot_table(index="Nationality", columns="Handedness",
               values="Sample")

TypeError: pivot_table() missing 1 required positional argument: 'data'

In [382]:
pd.crosstab(data["Nationality"], data["Handedness"], margins=True)

Handedness,Left-handed,Right-handed,All
Nationality,,,
Japan,2,3,5
USA,1,4,5
All,3,7,10


crosstab的前两个参数可以是数组、Series或者数组列表

In [389]:
pd.crosstab([tips["time"], tips["day"]], tips["smoker"], margins=True)

smoker        No  Yes  All
time   day                
Dinner Fri     3    9   12
       Sat    45   42   87
       Sun    57   19   76
       Thur    1    0    1
Lunch  Fri     1    6    7
       Thur   44   17   61
All          151   93  244